# CS604 Colab launcher — Ana: which re-ranker is best? (no training of the model)

A fair comparison of re-rankers on the **all-class holdout** (`scripts/rerank_bakeoff.py`): the LoRA r24 no-extras checkpoint trained 144 min on 40 photos per class of all 10,000 classes, scored on the 50,000 held-out photos it never saw. The bank is its 400,000 training photos' features.

For each photo the model's top 5 classes are re-ranked by:

| Re-ranker | Signal added to log p |
|---|---|
| knn_mean3 | mean of the 3 best matches to the candidate's training photos (**current method**, CodaBench +1.40 / +0.85) |
| knn_max | best single match |
| knn_meanall | mean match to all of the class's photos |
| ncm | match to the class's mean feature |
| centres3 | best match to 3 k-means centres of the class |
| vote | weighted vote of the photo's 50 nearest training photos in the whole bank |
| learned linear | a learned weighting of all signals (log p, rank, gap, the six above, class spread, sibling flag, the other view's log p) |
| learned trees | gradient-boosted trees on the same signals |

Each also **+ balancing** (Van's `src/van_balance.py`, ported unchanged: Sinkhorn to the holdout's 5 photos per class), plus balancing alone. All are run on the plain view and on **zoom** (crop 0.75: the LoRA-at-336 setting, Wenqi 0.799).

Fair protocol for every method: settings (or the learned model) chosen on half the holdout classes, scored on the other half, then swapped; fixes and breaks counted. Also writes `linear_reranker.json`: the learned weights, readable, for applying to other checkpoints.

About 45 min on an A100: data ~15 min, bank ~13 min, holdout with zoom and the comparison ~15 min. Results: `CS604_RUNS/rerank_bakeoff/`. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
RUN_RECORD = ("ablation_allclass_holdout_lora_no_extras", "A_all_classes_144min")  # the all-class holdout run
ZOOM_CROP = 0.75  # LoRA at 336: crop 0.75 + zoom was best (Wenqi, 0.799)
OUT_NAME = "rerank_bakeoff"
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Run the comparison on the all-class holdout

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from scripts.pilot_sweep import build_pilot_root

PILOT_ROOT, _ = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 10000, 5, 0)  # the same 40 / 5 split the run trained on
folder, arm = RUN_RECORD
RUN_DIR = json.loads((Path(DRIVE_OUT) / folder / "arms" / f"{arm}.json").read_text())["run_dir"]
assert (Path(RUN_DIR) / "last.pt").is_file(), f"no checkpoint in {RUN_DIR}"
OUT = Path(DRIVE_OUT) / OUT_NAME
if not (OUT / "bakeoff.csv").is_file():
    # stream the script's output into the notebook, so an error shows its full message
    proc = subprocess.Popen([sys.executable, "-u", "scripts/rerank_bakeoff.py", "--run_dir", RUN_DIR,
                             "--pilot_root", str(PILOT_ROOT), "--out_dir", str(OUT), "--zoom", "--crop_pct", str(ZOOM_CROP)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        end_runtime(f"bake-off failed (exit {proc.returncode})")
        raise SystemExit("bake-off failed")


## 6. Results

`gain_points`: held-out gain over the model's own top-1 (plain or zoom). `acc_balanced`: the same after balancing. The learned weights are in `linear_reranker.json`.

In [ ]:
import pandas as pd

table = pd.read_csv(OUT / "bakeoff.csv")
display(table.drop(columns=["chosen"]))
lin = json.loads((OUT / "linear_reranker.json").read_text())
for base, d in lin.items():
    print(base, "learned linear weights:", {k: round(w, 2) for k, w in zip(d["signals"], d["weights"])})


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")